In [1]:
# Find all recent competitions
from statsbombpy import sb
import pandas as pd

comps = sb.competitions()
comps.sort_values("season_name", ascending=False)[["competition_id", "season_id", "competition_name", "season_name"]].head(5)

/Users/michaelfordham/Documents/GitHub/wm-data/.venv/lib/python3.15/site-packages/statsbombpy/api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,competition_name,season_name
76,53,315,UEFA Women's Euro,2025
21,223,282,Copa America,2024
73,55,282,UEFA Euro,2024
0,9,281,1. Bundesliga,2023/2024
72,131,281,Serie A Women,2023/2024


In [3]:
# Loads every match and event, and save it into a file
from statsbombpy import sb
import pandas as pd

matches = sb.matches(competition_id=53, season_id=315)

all_events = []
for match_id in matches["match_id"]:
    all_events.append(sb.events(match_id=match_id))

events = pd.concat(all_events, ignore_index=True)

events.to_pickle("weuro2025_events.pkl")
matches.to_pickle("weuro2025_matches.pkl")

In [ ]:
# Part 1: Filtering
import pandas as pd
events = pd.read_pickle("weuro2025_events.pkl")
matches = pd.read_pickle("weuro2025_matches.pkl")

events["x"] = events["location"].str[0]
events["y"] = events["location"].str[1]

# Question 1: How many goals were scored from shots?
shots = events[(events["type"] == "Shot") & (events["period"] != 5)]
goals = shots[shots["shot_outcome"] == "Goal"]

pd.DataFrame({
    "Total Shots": [shots.shape[0]],
    "Total Goals": [goals.shape[0]],
    "Goal Conversion Rate": [round(goals.shape[0] / shots.shape[0] * 100, 2)]
})

In [ ]:
# Part 1: Filtering
import pandas as pd
events = pd.read_pickle("weuro2025_events.pkl")
matches = pd.read_pickle("weuro2025_matches.pkl")

events["x"] = events["location"].str[0]
events["y"] = events["location"].str[1]

# Question 2: Who completed the most passes into the box?
passes = events[events["type"] == "Pass"].copy()
passes["end_x"] = passes["pass_end_location"].str[0]
passes["end_y"] = passes["pass_end_location"].str[1]

# Ensures only completed passes are counted (they don't have an outcome)
completed = passes["pass_outcome"].isna()

# Ensures the pass ends but didn't start in the box
ends_in_box = (passes["end_x"] >= 102) & (passes["end_y"].between(18, 62))
starts_in_box = (passes["x"] >= 102) & (passes["y"].between(18, 62))

# Filters the passes to completed ones, ending in the box, NOT starting in the box and from open play
passes_into_box = passes[completed & ends_in_box & ~starts_in_box & passes["pass_type"].isna()]

# Groups by player descending
passes_into_box.groupby("player").size().sort_values(ascending=False).head(5)

player
María Francesca Caldentey Oliver    26
Alexia Putellas Segura              18
Ingrid Filippa Angeldal             14
Aitana Bonmati Conca                14
Patricia Guijarro Gutiérrez         14
dtype: int64

In [24]:
# Part 2: Groupby and agg
import pandas as pd
events = pd.read_pickle("weuro2025_events.pkl")
matches = pd.read_pickle("weuro2025_matches.pkl")

# Question 3: Which team took the most shots per match?
shots = events[(events["type"] == "Shot") & (events["period"] != 5)]

by_team = shots.groupby("team").agg(
    total_shots=("id", "count"),
    matches=("match_id", "nunique")
)

by_team["per_match"] = (by_team["total_shots"] / by_team["matches"]).round(1)
by_team.sort_values("per_match", ascending=False)

,total_shots,matches,per_match
team,,,
Spain Women's,145,6,24.2
England Women's,103,6,17.2
France Women's,67,4,16.8
Germany Women's,83,5,16.6
Sweden Women's,65,4,16.2
Italy Women's,69,5,13.8
Switzerland Women's,52,4,13.0
WNT Finland,36,3,12.0
Iceland Women's,35,3,11.7


In [31]:
# Part 2: Groupby and agg

import pandas as pd
events = pd.read_pickle("weuro2025_events.pkl")
matches = pd.read_pickle("weuro2025_matches.pkl")

shots = events[(events["type"] == "Shot") & (events["period"] != 5)]
np_shots = shots[shots["shot_type"] != "Penalty"]
np_shots["is_goal"] = np_shots["shot_outcome"] == "Goal"
np_shots["goals_minus_npxg"] = np_shots["is_goal"] - np_shots["shot_statsbomb_xg"]

by_player = np_shots.groupby(["team", "player"]).agg(
    total_shots=("id", "count"),
    npxg=("shot_statsbomb_xg", "sum"),
    npgoals=("is_goal", "sum"),
    goals_minus_npxg=("goals_minus_npxg", "sum")
)

by_player.sort_values("npxg", ascending=False).round(2).head(5)

total_shots  npxg  npgoals  \
team            player                                                  
Spain Women's   Esther Gonzalez Rodríguez           24  4.02        4   
Sweden Women's  Emma Stina Blackstenius             20  2.98        3   
England Women's Alessia Russo                       15  2.41        2   
                Ella Toone                          10  2.32        2   
Italy Women's   Cristiana Girelli                   12  2.24        3   

                                           goals_minus_npxg  
team            player                                       
Spain Women's   Esther Gonzalez Rodríguez             -0.02  
Sweden Women's  Emma Stina Blackstenius                0.02  
England Women's Alessia Russo                         -0.41  
                Ella Toone                            -0.32  
Italy Women's   Cristiana Girelli                      0.76